# 03 · The ingestion pipeline

**Purpose** — the centrepiece. Watch one technician update travel the whole
pipeline, stage by stage, with every intermediate visible.

**Prerequisites** — 01 has been run. NIMs up (notebook 00), or hosted fallback
via `NVIDIA_API_KEY`.

```
capture -> transcribe -> extract (3 C's) -> resolve -> validate -> reconcile -> diff card
```

The LLM is used **only to read language**. It never decides state, never computes
hours, never resolves an op code — all of that is deterministic downstream, which
is why the output can be trusted and cited.

In [ ]:
import sys, os, json
sys.path.insert(0, os.path.abspath(".."))          # notebooks/ -> repo root
os.chdir(os.path.abspath(".."))
from dotenv import load_dotenv; load_dotenv()
print("repo root:", os.getcwd())

## Config — which endpoints are in use

In [ ]:
from app.nim.client import health
for svc, info in health().items():
    print(f"  {svc:7s} {info['mode']:7s} {info['model']}")

## Stage 1 — the raw update

Real workshop language: trade shorthand, clipped phrasing, measurements.

In [ ]:
TEXT = ("C/S intermittent grinding front end under braking. Road tested, confirmed "
        "concern. Pulled front wheels - inner pad on the LF worn to backing, rotor "
        "scored past minimum spec at 22.8mm. Recommend pads and discs both sides. "
        "Parts checked - rotors are a next-day order so RO's on parts hold. "
        "Rear's still to inspect.")
print(TEXT)

## Stage 2 — extraction to the 3 C's

Concern / Cause / Correction is the documentation standard on every repair order,
so this output drops straight into a DMS.

In [ ]:
from app.state import db as dbm
from app.pipeline.extract import extract
from app.nim.client import embed as nim_embed

con = dbm.connect()
known = dbm.all_ro_numbers(con)
TARGET = known[165]
e = extract(f"{TARGET}. {TEXT}", known, embed_fn=nim_embed)

print("concern     :", e.concern)
print("cause       :", e.cause)
print("verified    :", e.verified)
print("completed   :", e.completed)
print("pending     :", e.pending)
print("recommended :", e.correction)
print("parts       :", e.parts)
print("measurements:", e.measurements)
print("severity    :", e.severity, "  <- derived from 22.8 < 23.0, not asserted by the model")
print("state signal:", e.state_signal)
print("confidence  :", e.confidence)

## Stage 3 — what did NOT resolve

Anything the catalog cannot confirm becomes a **question**, never a guess.

In [ ]:
if e.unresolved:
    for q in e.clarifying_questions(): print(" -", q)
else:
    print("everything resolved to canonical op codes")

## Stage 4 — reconcile into the event log, and the diff card

In [ ]:
from datetime import datetime
from app.pipeline.run import run
NOW = datetime.fromisoformat(os.environ.get("ASOIA_NOW", "2026-09-24T17:45:00"))
res = run(con, text=f"{TARGET}. {TEXT}", actor_id="EMP020", at=NOW, embed_fn=nim_embed)
print(res.diff_card)

## What you should see

A diff card naming what changed: operations completed with hours against flat
rate, work recommended that needs authorisation, the out-of-spec measurement
raised as a safety item, the state moved to parts hold, and the promised-time
consequence. Any conflict with what the log already said is surfaced, not overwritten.

### Voice
Same pipeline, one extra stage in front.

In [ ]:
# from app.pipeline.run import run
# res = run(con, audio_path="data/audio/sample.wav", actor_id="EMP020", at=NOW)
# print(res.transcript.text); print(res.diff_card)
print("Pass audio_path= instead of text= to run the ASR stage first.")